# 🟡 Solution: Implement Contrastive Loss

InfoNCE loss used in CLIP and SimCLR

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def contrastive_loss(q, k, temperature=0.07):
    """
    InfoNCE / NT-Xent 对比损失 (CLIP, SimCLR 使用)。

    核心思想：给定一批查询-键对，每个 q[i] 应被拉近其匹配键 k[i]，
    远离所有其他键。等价于目标为 [0,1,...,N-1] 的交叉熵。

    公式: L = mean_i( -log( exp(q_i·k_i/τ) / Σ_j exp(q_i·k_j/τ) ) )

    参数:
        q: 查询嵌入, shape = (N, D), 假设已 L2 归一化
        k: 键嵌入, shape = (N, D), k[i] 是 q[i] 的正样本
        temperature: 温度 τ，越小分布越尖锐

    返回: 标量均值损失
    """
    # ---- Step 1: 计算相似度矩阵 ----
    # q @ k^T: (N, D) @ (D, N) = (N, N)
    # logits[i, j] = q[i] · k[j] / τ
    # 对角线 logits[i, i] 是正样本对的相似度
    # 非对角线是负样本对的相似度
    # 除以 temperature: τ 越小，分布越尖锐，模型越「自信」
    logits = (q @ k.T) / temperature  # (N, N)

    N = q.shape[0]

    # ---- Step 2: 目标标签 ----
    # 目标是第 i 个查询匹配第 i 个键，即对角线
    targets = torch.arange(N, device=q.device)  # [0, 1, 2, ..., N-1]

    # ---- Step 3: 计算 log(Σ exp(logits)) ----
    # log_sum_exp 对每行计算: log(Σ_j exp(logits[i, j]))
    # 这是 softmax 的分母的 log
    # 这里用 torch.exp + sum + log 实现，也可以用 logsumexp
    log_sum_exp = torch.log(torch.exp(logits).sum(dim=-1))  # (N,)

    # ---- Step 4: 计算 log 概率 ----
    # log_p[i] = logits[i, targets[i]] - log_sum_exp[i]
    #          = log(exp(logits[i,i]) / Σ_j exp(logits[i,j]))
    #          = log(softmax(logits)[i, targets[i]])
    # 这就是交叉熵的 log 概率部分
    log_p = logits[torch.arange(N), targets] - log_sum_exp  # (N,)

    # ---- Step 5: 取负均值 ----
    return -log_p.mean()

In [ ]:
torch.manual_seed(0)
N, D = 8, 16

# Perfect alignment: q == k (normalized)
v = torch.randn(N, D)
q = v / v.norm(dim=-1, keepdim=True)
k = q.clone()
loss_perfect = contrastive_loss(q, k)
print(f"Perfect alignment loss: {loss_perfect:.4f}  (should be near log(1/N) = {-torch.log(torch.tensor(N, dtype=torch.float)):.4f})")

# Random embeddings should give higher loss
q_rand = torch.randn(N, D)
q_rand = q_rand / q_rand.norm(dim=-1, keepdim=True)
k_rand = torch.randn(N, D)
k_rand = k_rand / k_rand.norm(dim=-1, keepdim=True)
loss_rand = contrastive_loss(q_rand, k_rand)
print(f"Random embeddings loss:  {loss_rand:.4f}  (should be higher)")

In [ ]:
from torch_judge import check
check("contrastive_loss")

## 📝 核心思路总结

1. **InfoNCE = 对角线交叉熵**：`logits = q @ k.T / τ`，目标是 `[0,1,...,N-1]`，等价于让每个 q[i] 最匹配自己的 k[i]。
2. **温度 τ 的作用**：τ 小 → 分布尖锐 → 模型需要更高的置信度区分正负样本；τ 大 → 分布平滑 → 允许更多模糊。
3. **log-sum-exp 是数值稳定的关键**：`logsumexp(x) = max(x) + log(Σ exp(x - max(x)))`，直接 log(Σ exp(x)) 会溢出。
4. **批内负样本**：每个样本的负样本就是同 batch 的其他 N-1 个样本，batch 越大负样本越多，对比学习效果越好。